In [116]:
# ─────────────────────────────────────────────────────────────────
# UTILITIES  (unchanged from original)
# ─────────────────────────────────────────────────────────────────
def _strip_fences(text: str) -> str:
    """Remove markdown code fences from LLM output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$",          "", text)
    return text.strip()


def _safe_parse_list(raw: str, label: str) -> list:
    try:
        result = json.loads(raw)
        return result if isinstance(result, list) else []
    except (json.JSONDecodeError, ValueError) as e:
        print(f"[{label}] JSON parse error: {e} — attempting truncation recovery")
        # NEW: show the part near the error position
        if hasattr(e, 'pos') and e.pos:
            start = max(0, e.pos - 50)
            end = min(len(raw), e.pos + 50)
            print(f"[{label}] Error context: ...{raw[start:end]}...")
        # Your existing recovery logic...
        last_valid = raw.rfind("\n  }")
        if last_valid != -1:
            recovered = raw[: last_valid + 4] + "\n]"
            try:
                result = json.loads(recovered)
                result = result if isinstance(result, list) else []
                print(f"[{label}] recovered {len(result)} items")
                return result
            except (json.JSONDecodeError, ValueError):
                pass
        print(f"[{label}] recovery failed — returning []")
        # NEW: log the tail of the raw response to see if it's cut off
        print(f"[{label}] Raw response tail: ...{raw[-200:]}")
        return []

In [117]:
# ─────────────────────────────────────────────────────────────────
# Domain rules
# ─────────────────────────────────────────────────────────────────
from pydantic import BaseModel, Field, model_validator
MONOTONIC_DECREASING = {   # must DECREASE as credit score increases
    "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE",
    "DEFAULT_PROBABILITY_RECORDED_AS_HISTORICAL_NONREPAYMENT_RATE",
    "DOWN_PAYMENT_REQUIRED_AS_UPFRONT_BORROWER_CASH_CONTRIBUTION",
}

MONOTONIC_INCREASING = {   # must INCREASE as credit score increases
    "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD",
    "CREDIT_LIMIT_MULTIPLIER_APPLIED_AS_INCOME_BASED_BORROWING_CEILING_FACTOR",
    "MAX_DTI_PERMITTED_AS_MONTHLY_DEBT_BURDEN_CEILING",
}

TIER_ORDER = {
    "deep subprime": 1,
    "subprime":      2,
    "near prime":    3,
    "prime":         4,
    "super prime":   5,
}

LOGICAL_INCONSISTENCY_SYSTEM_PROMPT = """
You are a logical consistency checker for credit score knowledge graph triples.
Your ONLY job is to identify logical contradictions that exist WITHIN the provided
triples themselves — no external database, no domain facts, no outside knowledge.

A logical inconsistency means two or more triples directly contradict each other
by their own content, like saying something is both hot and cold at the same time.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INPUT STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
You will receive three pre-grouped sets of triples:

1. "same_subject_same_predicate"
   Triples sharing the same subject AND predicate but with different objects.
   Check if any two objects directly contradict each other.
   Type: DIRECT_CONFLICT

2. "same_predicate_different_scores"
   Triples sharing the same predicate but from different credit score subjects.
   Each predicate has a required direction:
     - DECREASING: higher credit score must NOT have a strictly HIGHER numeric value
    (equal values are acceptable and must NOT be flagged)
    - INCREASING: higher credit score must NOT have a strictly LOWER numeric value  
    (equal values are acceptable and must NOT be flagged)
     - TIER: higher credit score must have a higher or equal tier rank
       (deep subprime < subprime < near prime < prime < super prime)
   Extract the numeric score from the subject and the numeric value from the object,
   then check if any pair violates the required direction.
   Type: MONOTONICITY_VIOLATION or TIER_ORDER_VIOLATION

3. "all_triples"
   Complete flat list for reference.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INCONSISTENCY TYPES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

A. DIRECT_CONFLICT
   Same subject and predicate but two different objects simultaneously.
   Example: (600, HAS_INTEREST_RATE, 8%) AND (600, HAS_INTEREST_RATE, 15%)

B. MONOTONICITY_VIOLATION
   Higher credit score has a worse value than a lower credit score
   for a predicate that must strictly improve with score.
   Example: score 700 has interest rate 9.5% but score 580 has interest rate 7.2%

C. TIER_ORDER_VIOLATION
   A lower credit score has a higher risk tier than a higher credit score.
   Example: score 500 is "super prime" but score 750 is "subprime"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
- Only flag genuine contradictions within the triples themselves.
- Use NO external knowledge — only what the triples say.
- If no inconsistency exists in a group, produce no output for that group.
- If all groups are consistent, return [].
- For MONOTONICITY_VIOLATION: only flag when the higher score has a 
  strictly worse value than the lower score. If values are equal, 
  do NOT flag — equal values are valid and will be verified downstream.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OUTPUT FORMAT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Return a JSON array. Each object must have exactly these keys:
{
  "type":             "DIRECT_CONFLICT" |
                      "MONOTONICITY_VIOLATION" |
                      "TIER_ORDER_VIOLATION",
  "severity":         "HIGH" | "MEDIUM" | "LOW",
  "explanation":      "One sentence describing the contradiction.",
  "triples_involved": [
    {"subject": "...", "predicate": "...", "object": "..."},
    {"subject": "...", "predicate": "...", "object": "..."}
  ]
}

Severity guide:
  HIGH   — direct contradiction on the same subject (type A)
  MEDIUM — monotonicity or tier ordering violated across subjects (types B and C)
  LOW    — borderline or ambiguous

Output ONLY the JSON array. No markdown, no prose.
"""
from typing import Literal

# ─────────────────────────────────────────────────────────────────
# Pydantic models
# ─────────────────────────────────────────────────────────────────

class LogicalInconsistency(BaseModel):
    type: Literal[
        "DIRECT_CONFLICT",
        "MONOTONICITY_VIOLATION",
        "TIER_ORDER_VIOLATION",
    ]
    severity: Literal["HIGH", "MEDIUM", "LOW"]
    explanation: str
    triples_involved: List[Dict[str, str]]


class LogicalInconsistencyResult(BaseModel):
    inconsistencies: List[LogicalInconsistency]

    @model_validator(mode="before")
    @classmethod
    def wrap_if_list(cls, values):
        if isinstance(values, list):
            return {"inconsistencies": values}
        return values


# ─────────────────────────────────────────────────────────────────
# Group builder — organizes triples, LLM does the comparison
# ─────────────────────────────────────────────────────────────────

def _build_groups(canonicalized: list) -> dict:
    """
    Organizes canonicalized triples into three groups for the LLM to compare.
    No violation detection here — the LLM does all comparison and reasoning.
    """
    from collections import defaultdict

    triples = [item.get("canonical", {}) for item in canonicalized]

    ordered_predicates = MONOTONIC_DECREASING | MONOTONIC_INCREASING
    tier_predicate     = "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL"

    # ── Group 1: same subject + same predicate, different objects ─
    subj_pred_map = defaultdict(list)
    for t in triples:
        key = (
            str(t.get("subject",   "")).strip().lower(),
            str(t.get("predicate", "")).strip().upper(),
        )
        subj_pred_map[key].append(t)

    same_subject_same_predicate = [
        {
            "subject":   ts[0]["subject"],
            "predicate": ts[0]["predicate"],
            "triples":   ts,
        }
        for (_, __), ts in subj_pred_map.items()
        if len(ts) > 1
    ]

    # ── Group 2: same predicate, different score subjects ─────────
    # Includes ordered predicates and tier predicate
    # Annotated with required direction so LLM knows what to check
    pred_map = defaultdict(list)
    for t in triples:
        pred = str(t.get("predicate", "")).strip().upper()
        if pred in ordered_predicates or pred == tier_predicate:
            pred_map[pred].append(t)

    same_predicate_different_scores = []
    for pred, ts in pred_map.items():
        # Only include if there are multiple different score subjects
        subjects = {str(t.get("subject", "")).strip().lower() for t in ts}
        if len(subjects) < 2:
            continue

        if pred in MONOTONIC_DECREASING:
            direction = "DECREASING"
        elif pred in MONOTONIC_INCREASING:
            direction = "INCREASING"
        else:
            direction = "TIER"

        same_predicate_different_scores.append({
            "predicate":          pred,
            "required_direction": direction,
            "triples":            ts,
        })

    return {
        "same_subject_same_predicate":      same_subject_same_predicate,
        "same_predicate_different_scores":  same_predicate_different_scores,
        "all_triples":                      triples,
    }

# ─────────────────────────────────────────────────────────────────
# Pydantic models — fixed for OpenAI structured output compatibility
# ─────────────────────────────────────────────────────────────────

class TripleRef(BaseModel):
    subject:   str
    predicate: str
    object:    str


class LogicalInconsistency(BaseModel):
    type: Literal[
        "DIRECT_CONFLICT",
        "MONOTONICITY_VIOLATION",
        "TIER_ORDER_VIOLATION",
    ]
    severity: Literal["HIGH", "MEDIUM", "LOW"]
    explanation: str
    triples_involved: List[TripleRef]


class LogicalInconsistencyResult(BaseModel):
    inconsistencies: List[LogicalInconsistency]

    @model_validator(mode="before")
    @classmethod
    def wrap_if_list(cls, values):
        if isinstance(values, list):
            return {"inconsistencies": values}
        return values
# ─────────────────────────────────────────────────────────────────
# STEP 3 — detect_logical_inconsistencies
# ─────────────────────────────────────────────────────────────────

def detect_logical_inconsistencies(state: dict) -> dict:
    canonicalized = state.get("canonicalized_triples", [])
    if not canonicalized:
        state["logical_inconsistencies"] = []
        return state

    groups = _build_groups(canonicalized)

    # Skip LLM if no candidates in any group
    if (not groups["same_subject_same_predicate"] and
        not groups["same_predicate_different_scores"]):
        print("[detect_logical_inconsistencies] no candidates — skipping LLM")
        state["logical_inconsistencies"] = []
        return state

    messages = [
        SystemMessage(content=LOGICAL_INCONSISTENCY_SYSTEM_PROMPT),
        HumanMessage(content=json.dumps(groups, indent=2)),
    ]

    inconsistencies = []
    try:
        chain  = llm_reasoning_gemini.with_structured_output(LogicalInconsistencyResult)
        result = chain.invoke(messages)
        inconsistencies = [i.model_dump() for i in result.inconsistencies]
        print(f"[detect_logical_inconsistencies] {len(inconsistencies)} inconsistency/ies")

    except Exception as e:
        print(f"[detect_logical_inconsistencies] structured output failed: {e} — trying other fallback")
        try:
            raw_response = llm_reasoning_llama.invoke(messages)
            raw_text     = _strip_fences(raw_response.content)
            raw_list     = _safe_parse_list(raw_text, "detect_logical_inconsistencies")
            for item in raw_list:
                inconsistencies.append({
                    "type":             item.get("type",             "DIRECT_CONFLICT"),
                    "severity":         item.get("severity",         "MEDIUM"),
                    "explanation":      item.get("explanation",      ""),
                    "triples_involved": item.get("triples_involved", []),
                })
            print(f"[detect_logical_inconsistencies] GLM fallback: {len(inconsistencies)} inconsistency/ies")
        except Exception as e2:
            print(f"[detect_logical_inconsistencies] fallback also failed: {e2}")

    for inc in inconsistencies:
        inc["label"] = "logically_inconsistent"
        print(f"  → [{inc['severity']}] {inc['type']}: {inc['explanation']}")

    state["logical_inconsistencies"] = inconsistencies
    return state

In [118]:
import json

# ── paste your existing imports and LLM instances here ──
# from your_module import (
#     llm_reasoning_gemini, llm_reasoning_glm,
#     detect_logical_inconsistencies,
#     MONOTONIC_DECREASING, MONOTONIC_INCREASING, TIER_ORDER,
#     _strip_fences, _safe_parse_list,
#     LogicalInconsistency, LogicalInconsistencyResult,
# )

# ─────────────────────────────────────────────────────────────────
# TEST TRIPLES — injected directly as canonicalized triples
# covering all three inconsistency types
#
# _expected_type : the inconsistency type this triple is part of
#                  None means this triple is consistent (control)
# _note          : description of what this triple is testing
# ─────────────────────────────────────────────────────────────────

TEST_CANONICALIZED = [

    # ── DIRECT_CONFLICT cases ────────────────────────────────────
    # Same subject, same predicate, two different objects
    # Expected: 2 DIRECT_CONFLICT inconsistencies

    {
        "original": {"subject": "600", "predicate": "has risk tier", "object": "prime"},
        "canonical": {"subject": "600", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "prime"},
        "match_confidence": "HIGH",
        "_expected_type": "DIRECT_CONFLICT",
        "_note": "Score 600 assigned prime — conflicts with deep subprime below",
    },
    {
        "original": {"subject": "600", "predicate": "has risk tier", "object": "deep subprime"},
        "canonical": {"subject": "600", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "deep subprime"},
        "match_confidence": "HIGH",
        "_expected_type": "DIRECT_CONFLICT",
        "_note": "Score 600 assigned deep subprime — conflicts with prime above",
    },
    {
        "original": {"subject": "700", "predicate": "has interest rate", "object": "8.31%"},
        "canonical": {"subject": "700", "predicate": "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE", "object": "8.31%"},
        "match_confidence": "HIGH",
        "_expected_type": "DIRECT_CONFLICT",
        "_note": "Score 700 assigned 8.31% — conflicts with 15% below",
    },
    {
        "original": {"subject": "700", "predicate": "has interest rate", "object": "15%"},
        "canonical": {"subject": "700", "predicate": "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE", "object": "15%"},
        "match_confidence": "HIGH",
        "_expected_type": "DIRECT_CONFLICT",
        "_note": "Score 700 assigned 15% — conflicts with 8.31% above",
    },

    # ── MONOTONICITY_VIOLATION cases ─────────────────────────────
    # Higher score has worse value for an ordered predicate
    # Expected: 3 MONOTONICITY_VIOLATION inconsistencies

    # Interest rate — decreasing: score 750 should have lower rate than 580
    {
        "original": {"subject": "580", "predicate": "has interest rate", "object": "11.91%"},
        "canonical": {"subject": "580", "predicate": "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE", "object": "11.91%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 580 rate 11.91% — lower score, used to show 750 violates",
    },
    {
        "original": {"subject": "750", "predicate": "has interest rate", "object": "14.5%"},
        "canonical": {"subject": "750", "predicate": "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE", "object": "14.5%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 750 rate 14.5% — higher score but worse rate than 580",
    },

    # Approval odds — increasing: score 800 should have higher odds than 620
    {
        "original": {"subject": "620", "predicate": "has approval odds", "object": "55%"},
        "canonical": {"subject": "620", "predicate": "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD", "object": "55%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 620 approval odds 55% — lower score, used to show 800 violates",
    },
    {
        "original": {"subject": "800", "predicate": "has approval odds", "object": "30%"},
        "canonical": {"subject": "800", "predicate": "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD", "object": "30%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 800 approval odds 30% — higher score but worse odds than 620",
    },

    # Default probability — decreasing: score 740 should have lower prob than 400
    {
        "original": {"subject": "400", "predicate": "has default probability", "object": "28.7%"},
        "canonical": {"subject": "400", "predicate": "DEFAULT_PROBABILITY_RECORDED_AS_HISTORICAL_NONREPAYMENT_RATE", "object": "28.7%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 400 default prob 28.7% — lower score, used to show 740 violates",
    },
    {
        "original": {"subject": "740", "predicate": "has default probability", "object": "35%"},
        "canonical": {"subject": "740", "predicate": "DEFAULT_PROBABILITY_RECORDED_AS_HISTORICAL_NONREPAYMENT_RATE", "object": "35%"},
        "match_confidence": "HIGH",
        "_expected_type": "MONOTONICITY_VIOLATION",
        "_note": "Score 740 default prob 35% — higher score but worse prob than 400",
    },

    # ── TIER_ORDER_VIOLATION cases ────────────────────────────────
    # Lower score has higher tier than higher score
    # Expected: 2 TIER_ORDER_VIOLATION inconsistencies

    {
        "original": {"subject": "350", "predicate": "has risk tier", "object": "super prime"},
        "canonical": {"subject": "350", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "super prime"},
        "match_confidence": "HIGH",
        "_expected_type": "TIER_ORDER_VIOLATION",
        "_note": "Score 350 assigned super prime — lower score cannot have higher tier than 800",
    },
    {
        "original": {"subject": "800", "predicate": "has risk tier", "object": "subprime"},
        "canonical": {"subject": "800", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "subprime"},
        "match_confidence": "HIGH",
        "_expected_type": "TIER_ORDER_VIOLATION",
        "_note": "Score 800 assigned subprime — higher score has lower tier than 350",
    },
    {
        "original": {"subject": "500", "predicate": "has risk tier", "object": "prime"},
        "canonical": {"subject": "500", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "prime"},
        "match_confidence": "HIGH",
        "_expected_type": "TIER_ORDER_VIOLATION",
        "_note": "Score 500 assigned prime — lower score cannot have higher tier than 660",
    },
    {
        "original": {"subject": "660", "predicate": "has risk tier", "object": "near prime"},
        "canonical": {"subject": "660", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "near prime"},
        "match_confidence": "HIGH",
        "_expected_type": "TIER_ORDER_VIOLATION",
        "_note": "Score 660 assigned near prime — higher score has lower tier than 500",
    },

    # ── EQUAL VALUES — must NOT be flagged ───────────────────────
    # Same value for different scores is acceptable
    # Expected: 0 inconsistencies from these

    {
        "original": {"subject": "300", "predicate": "has approval odds", "object": "8%"},
        "canonical": {"subject": "300", "predicate": "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD", "object": "8%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 300 approval odds 8% — equal to 345, should not be flagged",
    },
    {
        "original": {"subject": "345", "predicate": "has approval odds", "object": "8%"},
        "canonical": {"subject": "345", "predicate": "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD", "object": "8%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 345 approval odds 8% — equal to 300, should not be flagged",
    },
    {
        "original": {"subject": "300", "predicate": "has down payment", "object": "22.1%"},
        "canonical": {"subject": "300", "predicate": "DOWN_PAYMENT_REQUIRED_AS_UPFRONT_BORROWER_CASH_CONTRIBUTION", "object": "22.1%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 300 down payment 22.1% — equal to 395, should not be flagged",
    },
    {
        "original": {"subject": "395", "predicate": "has down payment", "object": "22.1%"},
        "canonical": {"subject": "395", "predicate": "DOWN_PAYMENT_REQUIRED_AS_UPFRONT_BORROWER_CASH_CONTRIBUTION", "object": "22.1%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 395 down payment 22.1% — equal to 300, should not be flagged",
    },

    # ── CONSISTENT TRIPLES — control group ───────────────────────
    # Correct triples that should produce no inconsistencies
    # Expected: 0 inconsistencies from these

    {
        "original": {"subject": "580", "predicate": "has risk tier", "object": "subprime"},
        "canonical": {"subject": "580", "predicate": "RISK_TIER_CLASSIFIED_AS_BORROWER_DANGER_LEVEL", "object": "subprime"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 580 subprime — correct, should not be flagged",
    },
    {
        "original": {"subject": "660", "predicate": "has interest rate", "object": "9.49%"},
        "canonical": {"subject": "660", "predicate": "INTEREST_RATE_CHARGED_AS_ANNUAL_BORROWING_COST_PERCENTAGE", "object": "9.49%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 660 rate 9.49% — consistent with score 580 rate 11.91%",
    },
    {
        "original": {"subject": "740", "predicate": "has approval odds", "object": "89%"},
        "canonical": {"subject": "740", "predicate": "APPROVAL_ODDS_ESTIMATED_AS_LENDER_ACCEPTANCE_LIKELIHOOD", "object": "89%"},
        "match_confidence": "HIGH",
        "_expected_type": None,
        "_note": "Score 740 approval odds 89% — correct, should not be flagged",
    },
]

# ─────────────────────────────────────────────────────────────────
# EXPECTED SUMMARY
# ─────────────────────────────────────────────────────────────────
EXPECTED = {
    "DIRECT_CONFLICT":        2,   # (600 tier), (700 interest rate)
    "MONOTONICITY_VIOLATION": 3,   # interest rate, approval odds, default prob
    "TIER_ORDER_VIOLATION":   2,   # (350 vs 800), (500 vs 660)
    "false_positives":        0,   # equal values and control group must not be flagged
}

# ─────────────────────────────────────────────────────────────────
# RUN TEST
# ─────────────────────────────────────────────────────────────────

def run_test():
    print("=" * 60)
    print("  LOGICAL INCONSISTENCY DETECTION — ISOLATION TEST")
    print("=" * 60)
    print(f"  Total triples injected : {len(TEST_CANONICALIZED)}")
    print(f"  Expected DIRECT_CONFLICT        : {EXPECTED['DIRECT_CONFLICT']}")
    print(f"  Expected MONOTONICITY_VIOLATION : {EXPECTED['MONOTONICITY_VIOLATION']}")
    print(f"  Expected TIER_ORDER_VIOLATION   : {EXPECTED['TIER_ORDER_VIOLATION']}")
    print(f"  Expected false positives        : {EXPECTED['false_positives']}")
    print()

    # Strip meta keys before passing to the function
    clean = [
        {k: v for k, v in t.items() if not k.startswith("_")}
        for t in TEST_CANONICALIZED
    ]

    # Build the state dict the function expects
    state = {"canonicalized_triples": clean}

    # Run the detection function
    state = detect_logical_inconsistencies(state)

    inconsistencies = state.get("logical_inconsistencies", [])

    # ── Print results ─────────────────────────────────────────────
    print()
    print("=" * 60)
    print(f"  RESULTS — {len(inconsistencies)} inconsistency/ies detected")
    print("=" * 60)

    if not inconsistencies:
        print("  No inconsistencies detected.")
    else:
        for i, inc in enumerate(inconsistencies, 1):
            print(f"\n  [{i}] Type     : {inc['type']}")
            print(f"      Severity : {inc['severity']}")
            print(f"      Explanation : {inc['explanation']}")
            print(f"      Triples involved:")
            for t in inc.get("triples_involved", []):
                print(f"        ({t.get('subject')}, {t.get('predicate')}, {t.get('object')})")

    # ── Evaluate against expected ─────────────────────────────────
    print()
    print("=" * 60)
    print("  EVALUATION vs EXPECTED")
    print("=" * 60)

    counts = {
        "DIRECT_CONFLICT":        0,
        "MONOTONICITY_VIOLATION": 0,
        "TIER_ORDER_VIOLATION":   0,
    }
    for inc in inconsistencies:
        t = inc.get("type", "")
        if t in counts:
            counts[t] += 1

    # Triples that should not be flagged
    safe_subjects = {"300", "345", "395", "580", "660", "740"}
    false_positives = 0
    for inc in inconsistencies:
        for t in inc.get("triples_involved", []):
            subj = str(t.get("subject", "")).strip()
            # A false positive is flagging a triple from the safe group
            # that is not also part of a genuine inconsistency triple
            note = next(
                (x["_note"] for x in TEST_CANONICALIZED
                 if str(x["canonical"].get("subject", "")).strip() == subj
                 and x["_expected_type"] is None),
                None
            )
            if note and inc["type"] not in ("DIRECT_CONFLICT", "MONOTONICITY_VIOLATION", "TIER_ORDER_VIOLATION"):
                false_positives += 1

    all_correct = True
    for type_name, expected_count in EXPECTED.items():
        if type_name == "false_positives":
            got = false_positives
        else:
            got = counts.get(type_name, 0)
        status = "✓" if got == expected_count else "✗"
        if got != expected_count:
            all_correct = False
        print(f"  {status}  {type_name:<35} expected={expected_count}  got={got}")

    print()
    if all_correct:
        print("  RESULT: All expected inconsistencies detected correctly.")
    else:
        print("  RESULT: Some mismatches — review output above.")
    print("=" * 60)


if __name__ == "__main__":
    run_test()

  LOGICAL INCONSISTENCY DETECTION — ISOLATION TEST
  Total triples injected : 21
  Expected DIRECT_CONFLICT        : 2
  Expected MONOTONICITY_VIOLATION : 3
  Expected TIER_ORDER_VIOLATION   : 2
  Expected false positives        : 0

[detect_logical_inconsistencies] 6 inconsistency/ies
  → [HIGH] DIRECT_CONFLICT: The credit score 600 is simultaneously classified as both 'prime' and 'deep subprime'.
  → [HIGH] DIRECT_CONFLICT: The credit score 700 is assigned two different interest rates of 8.31% and 15% simultaneously.
  → [MEDIUM] TIER_ORDER_VIOLATION: A lower credit score (350) is assigned a higher risk tier (super prime) than a significantly higher credit score (800) assigned as subprime.
  → [MEDIUM] MONOTONICITY_VIOLATION: Higher credit score 800 has lower approval odds (30%) than the lower credit score 740 (89%) for a predicate that must increase with the score.
  → [MEDIUM] MONOTONICITY_VIOLATION: Higher credit score 740 has a higher default probability (35%) than the lower cred